# TensorFlow / Keras Common Syntax and Functional API

## 1. Basic Imports

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
```

You can call layers using the full form:

```python
tf.keras.layers.Conv2D(...)
```

or the shorter form if you already imported `layers`:

```python
layers.Conv2D(...)
```

## 2. Sequential API

Use the `Sequential` API when the model is a simple straight chain of layers:

```text
Input -> Conv -> BatchNorm -> ReLU -> Pool -> Flatten -> Dense
```

Template:

```python
model = keras.Sequential([
    layers.Input(shape=(64, 64, 3)),

    layers.Conv2D(32, kernel_size=3, strides=1, padding="same"),
    layers.BatchNormalization(axis=-1),
    layers.ReLU(),

    layers.MaxPool2D(pool_size=2, strides=2),

    layers.Flatten(),
    layers.Dense(1, activation="sigmoid")
])
```

In `Sequential([...])`, each layer is an element of a Python list, so remember to put a comma after each layer.

## 3. Basic Functional API

The Functional API builds a model by creating an input tensor, passing that tensor through layers, and then wrapping the input and output tensors into a model.

```python
inputs = keras.Input(shape=(64, 64, 3))

x = layers.Conv2D(32, 3, padding="same")(inputs)
x = layers.BatchNormalization(axis=-1)(x)
x = layers.ReLU()(x)
x = layers.MaxPool2D(2)(x)
x = layers.Flatten()(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=inputs, outputs=outputs)
```

The line:

```python
x = layers.Conv2D(32, 3)(x)
```

means:

```text
Create a Conv2D layer and apply it to tensor x.
```

## 4. Input Syntax

```python
inputs = keras.Input(shape=(64, 64, 3))
```

Do not include the batch size inside `shape`.

TensorFlow/Keras usually uses the **channels-last** format:

```text
(batch, height, width, channels)
```

Therefore, an RGB image of size `64 x 64` has the input shape:

```python
(64, 64, 3)
```

A grayscale MNIST image has the input shape:

```python
(28, 28, 1)
```

## 5. Conv2D

`Conv2D` creates convolution kernels and applies them to the input tensor to produce an output tensor. If bias and activation are used, the bias is added first, then the activation is applied.

Common full syntax:

```python
layers.Conv2D(
    filters=32,
    kernel_size=(3, 3),
    strides=(1, 1),
    padding="same",
    activation="relu"
)
```

Shorter syntax:

```python
layers.Conv2D(32, 3, strides=1, padding="same", activation="relu")
```

Parameter meanings:

| Parameter     | Meaning                                               |
| ------------- | ----------------------------------------------------- |
| `filters`     | Number of filters, also the number of output channels |
| `kernel_size` | Size of the convolution filter                        |
| `strides`     | Step size of the filter                               |
| `padding`     | `"valid"` or `"same"`                                 |
| `activation`  | Activation function after convolution                 |

Example:

```python
layers.Conv2D(32, 7, strides=1, padding="valid")
```

## 6. Padding

### ZeroPadding2D

```python
layers.ZeroPadding2D(padding=(3, 3))
```

or:

```python
layers.ZeroPadding2D(padding=3)
```

Example:

```text
64 x 64 x 3 -> 70 x 70 x 3
```

because padding `3` adds 3 pixels to the top, bottom, left, and right sides.

### Padding Inside Conv2D

```python
layers.Conv2D(32, 3, padding="valid")
```

`"valid"` means no additional padding is applied inside the convolution layer.

```python
layers.Conv2D(32, 3, padding="same")
```

`"same"` usually keeps the spatial size unchanged when `strides=1`.

## 7. BatchNormalization

Batch Normalization helps make activations have a mean close to `0` and a standard deviation close to `1`. During training, it uses the mean and variance of the current mini-batch. During inference, it uses moving statistics learned during training.

For TensorFlow CNN tensors:

```text
(m, H, W, C)
```

the channel axis is the last axis, so use:

```python
layers.BatchNormalization(axis=-1)
```

or equivalently:

```python
layers.BatchNormalization(axis=3)
```

Common pattern:

```python
layers.Conv2D(32, 3, padding="same"),
layers.BatchNormalization(axis=-1),
layers.ReLU(),
```

Important idea:

```text
BatchNorm axis is the feature/channel axis to keep.
It is not the axis to reduce like np.mean(axis=...).
```

For Dense layer outputs:

```text
(m, n_units)
```

use:

```python
layers.BatchNormalization(axis=-1)
```

Do not use:

```python
layers.BatchNormalization(axis=0)
```

because axis `0` is the batch axis, not the feature axis.

## 8. Activation / ReLU

There are several common ways to apply ReLU.

Method 1:

```python
layers.ReLU()
```

Method 2:

```python
layers.Activation("relu")
```

Method 3, directly inside another layer:

```python
layers.Conv2D(32, 3, activation="relu")
layers.Dense(128, activation="relu")
```

If Batch Normalization is used, the common pattern is:

```text
Conv2D -> BatchNormalization -> ReLU
```

## 9. Pooling

`MaxPool2D` downsamples the height and width dimensions by taking the maximum value inside each local window.

```python
layers.MaxPool2D(pool_size=(2, 2), strides=(2, 2))
```

Shorter version:

```python
layers.MaxPool2D(2)
```

Equivalent naming:

```python
layers.MaxPooling2D(2)
```

Average pooling:

```python
layers.AveragePooling2D(pool_size=2, strides=2)
```

Global average pooling:

```python
layers.GlobalAveragePooling2D()
```

Example:

```text
8 x 8 x 128 -> 128
```

Pooling reduces the spatial dimensions, but usually keeps the number of channels unchanged.

## 10. Flatten, Dense, and Dropout

### Flatten

```python
layers.Flatten()
```

Example:

```text
8 x 8 x 64 -> 4096
```

### Dense Hidden Layer

```python
layers.Dense(128, activation="relu")
```

### Binary Classification Output

```python
layers.Dense(1, activation="sigmoid")
```

### Multi-Class Output as Logits

```python
layers.Dense(num_classes)
```

### Dropout

```python
layers.Dropout(0.5)
```

`Dropout(0.5)` randomly drops 50% of the activations during training.

## 11. Compile

After creating a model, use `compile()` to configure the optimizer, loss function, and metrics.

### Binary Classification

```python
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
```

### Multi-Class Classification with Integer Labels

Use this when labels are integers such as `0`, `1`, `2`, and so on.

```python
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
```

### Multi-Class Classification with One-Hot Labels

```python
model.compile(
    optimizer="adam",
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
```

### Optimizer with Custom Learning Rate

```python
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
```

## 12. Train, Evaluate, and Predict

### Train with NumPy Arrays

```python
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val)
)
```

### Train with a Dataset

```python
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)
```

### Evaluate

```python
loss, acc = model.evaluate(X_test, y_test)
```

### Predict

```python
preds = model.predict(X_test)
```

### Binary Prediction

```python
y_pred = (preds > 0.5).astype("int")
```

### Multi-Class Prediction from Logits

```python
logits = model.predict(X_test)
probs = tf.nn.softmax(logits, axis=-1)
y_pred = tf.argmax(probs, axis=-1)
```

### Model Summary

```python
model.summary()
```

## 13. Functional API with Skip Connection

This is one of the main reasons to use the Functional API. It can represent models that are not simple straight chains, such as ResNet-style blocks.

Example:

```python
inputs = keras.Input(shape=(32, 32, 64))

shortcut = inputs

x = layers.Conv2D(64, 3, padding="same")(inputs)
x = layers.BatchNormalization(axis=-1)(x)
x = layers.ReLU()(x)

x = layers.Conv2D(64, 3, padding="same")(x)
x = layers.BatchNormalization(axis=-1)(x)

x = layers.Add()([x, shortcut])
outputs = layers.ReLU()(x)

model = keras.Model(inputs=inputs, outputs=outputs)
```

The key line is:

```python
x = layers.Add()([x, shortcut])
```

It means:

```text
main path output + skip connection
```

## 14. Functional API with Multiple Branches

You can create two parallel branches and concatenate them.

```python
inputs = keras.Input(shape=(64, 64, 3))

branch1 = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
branch2 = layers.Conv2D(32, 5, padding="same", activation="relu")(inputs)

x = layers.Concatenate(axis=-1)([branch1, branch2])

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10)(x)

model = keras.Model(inputs=inputs, outputs=outputs)
```

If each branch has shape:

```text
(m, 64, 64, 32)
```

then after concatenation along the channel axis:

```text
(m, 64, 64, 64)
```

## 15. Functional API with Multiple Inputs

```python
image_input = keras.Input(shape=(64, 64, 3), name="image")
meta_input = keras.Input(shape=(10,), name="metadata")

x1 = layers.Conv2D(32, 3, activation="relu")(image_input)
x1 = layers.GlobalAveragePooling2D()(x1)

x2 = layers.Dense(32, activation="relu")(meta_input)

x = layers.Concatenate()([x1, x2])
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(
    inputs=[image_input, meta_input],
    outputs=outputs
)
```

Training:

```python
model.fit(
    [X_images, X_metadata],
    y_train,
    epochs=10,
    batch_size=32
)
```

## 16. Functional API with Multiple Outputs

```python
inputs = keras.Input(shape=(64, 64, 3))

x = layers.Conv2D(32, 3, activation="relu")(inputs)
x = layers.GlobalAveragePooling2D()(x)

class_output = layers.Dense(10, name="class_output")(x)
binary_output = layers.Dense(1, activation="sigmoid", name="binary_output")(x)

model = keras.Model(
    inputs=inputs,
    outputs=[class_output, binary_output]
)
```

Compile with multiple losses:

```python
model.compile(
    optimizer="adam",
    loss={
        "class_output": keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        "binary_output": "binary_crossentropy"
    },
    metrics={
        "class_output": ["accuracy"],
        "binary_output": ["accuracy"]
    }
)
```

## 17. CNN Template with Functional API

```python
def build_cnn(input_shape=(64, 64, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding="same")(inputs)
    x = layers.BatchNormalization(axis=-1)(x)
    x = layers.ReLU()(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization(axis=-1)(x)
    x = layers.ReLU()(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Conv2D(128, 3, padding="same")(x)
    x = layers.BatchNormalization(axis=-1)(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(num_classes)(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    return model
```

Compile:

```python
model = build_cnn(input_shape=(64, 64, 3), num_classes=10)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.summary()
```

## 18. HappyModel with Functional API

```python
def happyModel():
    inputs = keras.Input(shape=(64, 64, 3))

    x = layers.ZeroPadding2D(padding=(3, 3))(inputs)
    x = layers.Conv2D(filters=32, kernel_size=(7, 7), strides=(1, 1))(x)
    x = layers.BatchNormalization(axis=3)(x)
    x = layers.ReLU()(x)
    x = layers.MaxPool2D()(x)
    x = layers.Flatten()(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    return model
```

Shape flow:

```text
64 x 64 x 3
-> 70 x 70 x 3
-> 64 x 64 x 32
-> 64 x 64 x 32
-> 64 x 64 x 32
-> 32 x 32 x 32
-> 32768
-> 1
```

## 19. Common Syntax Mistakes

| Wrong                                  | Correct                                   |
| -------------------------------------- | ----------------------------------------- |
| `stride=1`                             | `strides=1`                               |
| `BactchNormalization`                  | `BatchNormalization`                      |
| `layers.activation.ReLU()`             | `layers.ReLU()`                           |
| `layers.Sigmoid()` for output          | `layers.Dense(1, activation="sigmoid")`   |
| Missing commas in `Sequential`         | Add `,` after each layer                  |
| `BatchNormalization(axis=0)` for Dense | `BatchNormalization(axis=-1)`             |
| Softmax output with `from_logits=True` | Remove softmax or set `from_logits=False` |

## 20. Short Summary

With TensorFlow/Keras, remember these three patterns.

### Sequential

```python
model = keras.Sequential([
    layers.Input(shape=(H, W, C)),
    layers.Conv2D(32, 3, padding="same"),
    layers.BatchNormalization(axis=-1),
    layers.ReLU(),
    layers.MaxPool2D(2),
    layers.Flatten(),
    layers.Dense(num_classes)
])
```

### Functional API

```python
inputs = keras.Input(shape=(H, W, C))

x = layers.Conv2D(32, 3, padding="same")(inputs)
x = layers.BatchNormalization(axis=-1)(x)
x = layers.ReLU()(x)

outputs = layers.Dense(num_classes)(x)

model = keras.Model(inputs=inputs, outputs=outputs)
```

### Training

```python
model.compile(optimizer="adam", loss=..., metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10, batch_size=32)
model.evaluate(X_test, y_test)
model.predict(X_test)
```

Final memory:

```text
Sequential is for straight-chain models.
Functional API is for graph-like models.
BatchNorm axis is the feature/channel axis to keep.
```
